# rl19_step

> Phase 19 (`specs/multiagent/phase19/`): the pieces `trainRL17` adds for Phase 19, all skipped at the defaults. The varied-threat mix (`threatDraws`: each board's liar count and lifetime from RNGs seeded by the step alone, so every arm and seed sees the same draws at a step; `stepGroups`); the rollout world's options from the config (`world19`: contest slots, half-life, the all-sharing penalty, the lifetime); the step's rollouts (`playStep`, exactly Phase 17's at the defaults); arm 4's per-agent records carried across steps (`RecordBook`); and the answer halt (`answerStats`, `haltReason19`, FR-A8) beside `FIX18`'s KL rule and the invalid-reply halt.

In [ ]:
#| default_exp rl19_step

In [ ]:
#| hide
from nbdev.showdoc import *

## Board types, worlds and rollouts

In [ ]:
#| export
import json
from pathlib import Path

import numpy as np

from explRL.clue_roll import rollout
from explRL.colony_agents import scriptedFaults
from explRL.contest import Records, boardType, honestAnswers
from explRL.rl_control import haltReason
from explRL.rl17_update import boardGroups

SEQ = 6                                  # boards per sequence of per-agent records (FR-A7)


def threatDraws(cfg, s):
    """[(liars of 16, lifetime)] per distinct board of step s from cfg.threat = (liar counts, lifetimes): two RNGs seeded by
    the step only ([s, 0] for liars, [s, 1] for lifetimes), so arms 2 and 3 draw the same lifetimes and every run the same types."""
    counts, lives = cfg.threat
    a = np.random.default_rng([s, 0]).integers(len(counts), size=cfg.seedsPerStep)
    b = np.random.default_rng([s, 1]).integers(len(lives), size=cfg.seedsPerStep)
    return [(int(counts[x]), int(lives[y])) for x, y in zip(a, b)]


def stepGroups(cfg, s):
    """[(N, nLiar, life, seeds, keys)]: the step's board groups. Without a threat, boardGroups' (life None: the world's) with
    keys (N, nLiar, board); with one, boards of one type (N, liars scaled from 16 to N, lifetime) in a group, keys (N, nLiar, j, copy)
    for distinct board j and GRPO copy (each seed repeated `group` times, the distinct seeds split by mixN as boardGroups splits
    them). The liar count in the key keeps a record sequence's liar slots fixed (records follow keys, RecordBook)."""
    if getattr(cfg, 'threat', None) is None:
        return [(N, nl, None, seeds, [(N, nl, b) for b in range(len(seeds))]) for N, nl, seeds in boardGroups(cfg, s)]
    distinct, draws = [cfg.boardSeed0 + s * cfg.seedsPerStep + j for j in range(cfg.seedsPerStep)], threatDraws(cfg, s)
    out, at = {}, 0
    for k, (N, share) in enumerate(cfg.mixN):
        n = len(distinct) - at if k == len(cfg.mixN) - 1 else int(round(cfg.seedsPerStep * share))
        for j in range(at, at + n):
            nl = int(round(draws[j][0] * N / 16))
            g = out.setdefault((N, nl, draws[j][1]), ([], []))
            g[0].extend([distinct[j]] * cfg.group)
            g[1].extend((N, nl, j, c) for c in range(cfg.group))
        at += n
    return [(N, nl, life, seeds, keys) for (N, nl, life), (seeds, keys) in out.items()]


def world19(cfg, life=None):
    "The rollout world's Phase 19 options from the config: contest slots and half-life, the penalty, the lifetime ({} at the defaults)."
    w = {}
    if getattr(cfg, 'contestSlots', None):
        w.update(contestSlots=cfg.contestSlots, halfLife=float(cfg.halfLife))
    if getattr(cfg, 'sharePenalty', 0.0):
        w['sharePenalty'] = float(cfg.sharePenalty)
    if life is not None:
        w['life'] = int(life)
    return w


def check19(cfg):
    "Refuse a Phase 19 configuration that cannot run: contest slots without scheme 'contest', or a penalty or answer halt without warmAnswers."
    w = dict(cfg.world)
    if (cfg.scheme == 'contest') != bool(cfg.contestSlots):
        raise ValueError("scheme 'contest' and contestSlots go together")
    if (cfg.sharePenalty or cfg.haltAnswers) and 'warmAnswers' not in w:
        raise ValueError('sharePenalty and haltAnswers need warmAnswers in the world (the speed check\'s WARM_ANSWERS)')
    return True


def playStep(cfg, s, lcfg, pol, book=None):
    """The step's rollouts [(LLMConfig, Rollout)]: lcfg(N, nLiar, life) per group; with `book` (records), each board's Records.
    At the defaults: trainRL17's rollouts exactly (boardGroups' groups, the world's lifetime, no records)."""
    out = []
    for N, nl, life, seeds, keys in stepGroups(cfg, s):
        c = lcfg(N, nl, life)
        out.append((c, rollout(c, seeds, pol, scriptedFaults(dict(cfg.world), nl), keepPrompts=True,
                               records=book.take(keys, N) if book is not None else None)))
    return out

## Records across steps and the answer halt

In [ ]:
#| export
class RecordBook:
    """Arm 4's per-agent records in training: one contest.Records per board slot of a step (its key: team size, liar count,
    distinct board, GRPO copy), carried to the next step with the same key, so a sequence keeps its liar slots (a key recurs
    when its board draws that liar count again; Records resets itself after 6 boards). `take` hands out fresh
    copies of the committed records (a rolled-back step replays from the same state); `commit` keeps what the step's
    rollouts added; `save` writes them with the step's checkpoint (records/step-<step>.json), and a resume loads that step's."""
    def __init__(self, out, start=0):
        self.dir, self.state, self.live = Path(out) / 'records', {}, {}
        if start:
            p = self.dir / f'step-{start:04d}.json'
            if not p.exists():
                raise FileNotFoundError(f'no records saved with the step {start} checkpoint ({p})')
            self.state = {tuple(k): v for k, v in json.loads(p.read_text())}

    def take(self, keys, N):
        recs = []
        for k in keys:
            r = Records(N, SEQ)
            if k in self.state:
                r.boards, r.cur = [list(map(dict, b)) for b in self.state[k]['boards']], list(map(dict, self.state[k]['cur']))
            recs.append(r)
        self.live.update(zip(map(tuple, keys), recs))
        return recs

    def commit(self):
        self.state.update({k: dict(boards=r.boards, cur=r.cur) for k, r in self.live.items()})   # keys not played this step keep theirs
        self.live = {}

    def save(self, step):
        self.dir.mkdir(parents=True, exist_ok=True)
        (self.dir / f'step-{step:04d}.json').write_text(json.dumps([[list(k), v] for k, v in sorted(self.state.items())]) + '\n')


def answerStats(cfg, gs):
    """The step's honest answers against the warm start's on the same board types: {answers, answerRef, answerRatio}
    ({} unless the run has the answer halt or the penalty)."""
    if not (getattr(cfg, 'haltAnswers', None) or getattr(cfg, 'sharePenalty', 0.0)):
        return {}
    tab = {tuple(int(x) for x in k): float(v) for k, v in dict(cfg.world)['warmAnswers']}
    a = sum(honestAnswers(bd, c.nLiar) for c, ro in gs for bd in ro.boards)
    ref = sum(tab[boardType(c.N, c.nLiar, bd.life)] for c, ro in gs for bd in ro.boards)
    return dict(answers=int(a), answerRef=float(ref), answerRatio=a / ref if ref else float('nan'))


def haltReason19(history, cfg):
    """haltReason (invalid replies; KL, FIX18's run of steps), then the answer halt: haltAnswers steps in a row whose honest
    answers fall below a third of the warm start's (answerRatio < 1/3). At haltAnswers None: haltReason exactly."""
    r, k = haltReason(history, cfg), getattr(cfg, 'haltAnswers', None)
    if r or not k:
        return r
    last = history[-k:]
    if len(last) == k and all(h.get('answerRatio', 1.0) < 1 / 3 for h in last):
        return f"answers per board below a third of the warm start's for {k} steps (steps {last[0]['step']}-{last[-1]['step']})"
    return None

## Tests

In [ ]:
# the threat mix: the same draws for every arm and seed at a step; arm 3's lifetimes are arm 2's; groups hold the step's boards
from explRL.rl19_grid import FIXED4, VARIED, RL19Config, rl19Config
from explRL.rl17_grid import RL17Config
from explRL.rl19_step import *
a2, a3 = RL19Config('m', 'o', threat=VARIED, mixN=((16, 1.0),), trainSeed=3), RL19Config('m', 'o', threat=FIXED4, mixN=((16, 1.0),))
for s in range(5):
    d2, d3 = threatDraws(a2, s), threatDraws(a3, s)
    assert d2 == threatDraws(RL19Config('x', 'y', threat=VARIED), s) and [L for _, L in d2] == [L for _, L in d3]
    assert {n for n, _ in d3} == {4} and {n for n, _ in d2} <= {0, 4, 6} and {L for _, L in d2} <= {15, 30}
    gs = stepGroups(a2, s)
    assert sum(len(x[3]) for x in gs) == a2.seedsPerStep * a2.group and all(len(x[3]) % a2.group == 0 for x in gs)
    assert all(x[0] == 16 and (x[1], x[2]) in set(d2) for x in gs) and len({k for x in gs for k in x[4]}) == 32
assert {n for s in range(10) for n, _ in threatDraws(a2, s)} == {0, 4, 6}
assert [(n, round(n * 8 / 16)) for n in (4, 6)] == [(4, 2), (6, 3)]          # liars scale with the team size
assert stepGroups(RL17Config('m', 'o', mixN=((4, 0.5), (8, 0.5)), liarShare=0.25), 2) == \
    [(N, nl, None, sd, [(N, nl, b) for b in range(len(sd))]) for N, nl, sd in boardGroups(RL17Config('m', 'o', mixN=((4, 0.5), (8, 0.5)), liarShare=0.25), 2)]
assert world19(RL17Config('m', 'o')) == {} == world19(RL19Config('m', 'o'))
assert world19(RL19Config('m', 'o', contestSlots='all', halfLife=2, sharePenalty=0.5), 15) == dict(contestSlots='all', halfLife=2.0, sharePenalty=0.5, life=15)
for bad in (RL19Config('m', 'o', scheme='contest'), RL19Config('m', 'o', contestSlots='guards'), RL19Config('m', 'o', haltAnswers=3)):
    try:
        check19(bad)
        raise AssertionError(bad)
    except ValueError:
        pass

In [ ]:
# test_penalty, its trainer half (AC-4): the answer halt fires after exactly 3 steps in a row below a third of the reference (a
# step at exactly a third does not count), not before, and not when the streak breaks; the invalid and KL halts come first
from types import SimpleNamespace
cfg = RL19Config('m', 'o', haltAnswers=3, haltKLRun=2)
H = lambda *r: [dict(step=i + 1, invalid=0.0, kl=0.0, answerRatio=x) for i, x in enumerate(r)]
assert haltReason19(H(0.2, 0.2), cfg) is None and haltReason19(H(0.5, 0.2, 0.2), cfg) is None
assert haltReason19(H(0.5, 0.2, 0.2, 0.2), cfg) == "answers per board below a third of the warm start's for 3 steps (steps 2-4)"
assert haltReason19(H(0.2, 0.2, 1 / 3), cfg) is None and haltReason19(H(0.2, 0.4, 0.2, 0.2), cfg) is None
assert haltReason19(H(0.2, 0.2, 0.2), RL19Config('m', 'o')) is None                 # off at the defaults
kl = [dict(h, kl=0.9) for h in H(0.2, 0.2, 0.2)]
assert haltReason19(kl, cfg).startswith('mean KL') and haltReason19(kl, RL17Config('m', 'o', haltKLRun=2)) == haltReason(kl, RL17Config('m', 'o', haltKLRun=2))
# answerStats: honest submissions over the step's boards against the table for their board types
board = lambda n, life: SimpleNamespace(submits=[(0.0, i, 0, None, 'wrong') for i in range(n)], life=life)
gs = [(SimpleNamespace(N=16, nLiar=4), SimpleNamespace(boards=[board(6, 15.0), board(4, 15.0)])),
      (SimpleNamespace(N=16, nLiar=0), SimpleNamespace(boards=[board(3, 30.0)]))]
cw = RL19Config('m', 'o', haltAnswers=3, world=(('warmAnswers', (((16, 4, 15), 6.0), ((16, 0, 30), 9.0))),))
st = answerStats(cw, gs)                      # honest: slots >= nLiar, so 2 + 0 + 3 = 5 answers against 6 + 6 + 9
assert st == dict(answers=5, answerRef=21.0, answerRatio=5 / 21) and answerStats(RL19Config('m', 'o'), gs) == {}

In [ ]:
# records across steps: take, commit, save and a resume give the same records; a rolled-back step (take without commit) adds nothing
import tempfile
from types import SimpleNamespace as NS
with tempfile.TemporaryDirectory() as d:
    book = RecordBook(d)
    fake = NS(netByAgent=lambda: np.arange(4.0), nAgents=4, contestLog=None)
    for step in range(1, 4):
        recs = book.take([(4, 0, 0), (4, 1, 0)], 4)
        if step == 2:                                 # a step played, rolled back and played again: the replay starts afresh
            for r in recs:
                r.addBoard(fake, range(1, 4))
            recs = book.take([(4, 0, 0), (4, 1, 0)], 4)
            assert len(recs[0].boards) == 1
        for r in recs:
            r.addBoard(fake, range(1, 4))
        book.commit()
        book.save(step)
    book.take([(4, 9, 0)], 4)[0].addBoard(fake, range(1, 4))      # a key played at one step only (another liar count) ...
    book.commit()
    book.take([(4, 0, 0)], 4)
    book.commit()
    assert len(book.state[(4, 9, 0)]['boards']) == 1                # ... keeps its record when later steps do not draw it
    again = RecordBook(d, 3).take([(4, 0, 0)], 4)[0]
    assert len(again.boards) == 3 and again.boards[2][3]['answers'] == 3.0 and again.line(0).count('board ') == 3
    assert again.line(1) == book.take([(4, 0, 0)], 4)[0].line(1)
    for _ in range(3):
        again.addBoard(fake, range(4))
    assert again.boards == []                              # six boards: the sequence ends and the record resets
    try:
        RecordBook(d, 5)
        raise AssertionError('resumed without records')
    except FileNotFoundError:
        pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()